In [9]:
import pyarrow.parquet as pq
from pathlib import Path

root = Path("intelligent-data-analysis-and-visualization/data")  # папка с parquet-файлами

for p in sorted(root.rglob("*.parquet")):
    pf = pq.ParquetFile(p)
    md = pf.metadata
    print(f"\n=== {p.name} | {p.stat().st_size/1e6:.1f} МБ | строк: {md.num_rows:,} | row groups: {md.num_row_groups}")
    for f in pf.schema_arrow:
        print(f"  {f.name}: {f.type}")
    head = next(pf.iter_batches(batch_size=3)).to_pandas()
    print(head.to_string(max_colwidth=40))

In [10]:
import pandas as pd
import pyarrow.parquet as pq

BIG = "data/iavd_incidents_raw_target.parquet"  # поправь путь

# 1) доля заполненности колонок по метаданным (без загрузки данных)
pf = pq.ParquetFile(BIG)
md = pf.metadata
nulls = {}
for rg in range(md.num_row_groups):
    for ci in range(md.num_columns):
        c = md.row_group(rg).column(ci)
        n = c.statistics.null_count if (c.statistics and c.statistics.has_null_count) else None
        nulls[c.path_in_schema] = None if (n is None or nulls.get(c.path_in_schema, 0) is None) \
            else nulls.get(c.path_in_schema, 0) + n
print("=== Заполненность колонок, % ===")
rows = md.num_rows
fill = {k: (None if v is None else round(100 * (1 - v / rows), 1)) for k, v in nulls.items()}
for k, v in sorted(fill.items(), key=lambda kv: (kv[1] is None, -(kv[1] or 0))):
    print(f"{v!s:>6}  {k}")

# 2) значения ключевых колонок
cols = ["Создан", "Вердикт", "verdict", "Критичность", "Критичность инцидента в источнике",
        "Группа устранения", "Источник инцидента", "Тип инцидента", "Наименование",
        "Количество инцидентов", "Время решения", "Период взятия в работу",
        "False Positive", "Ложное срабатывание", "Легитимность действия", "Белый список", "Спам"]
df = pq.read_table(BIG, columns=cols).to_pandas()

print("\n=== Период:", df["Создан"].min(), "—", df["Создан"].max(), "| строк:", len(df))
print("\n=== Вердикт x verdict ===")
print(pd.crosstab(df["Вердикт"], df["verdict"], dropna=False))

for c in ["Критичность", "Критичность инцидента в источнике", "Группа устранения", "Источник инцидента"]:
    print(f"\n=== {c} ===")
    print(df[c].value_counts(dropna=False))

print("\n=== Тип инцидента (топ-10) ===")
print(df["Тип инцидента"].value_counts(dropna=False).head(10))
print("\n=== Наименование: уникальных", df["Наименование"].nunique(), "; топ-10 ===")
print(df["Наименование"].value_counts().head(10))

for c in ["False Positive", "Ложное срабатывание", "Легитимность действия", "Белый список", "Спам"]:
    print(f"\n=== {c} x Вердикт ===")
    print(pd.crosstab(df["Вердикт"], df[c], dropna=False))

print("\n=== Числовые ===")
print(df[["Количество инцидентов", "Время решения", "Период взятия в работу"]].describe().round(1))

=== Заполненность колонок, % ===
 100.0  EntityId
 100.0  Создан
 100.0  Дата
 100.0  Вердикт
 100.0  Тип инцидента
 100.0  Критичность
 100.0  Пиктограмма IRP
 100.0  Группа устранения
 100.0  Фаза реагирования
 100.0  Источник инцидента
 100.0  Организация
 100.0  Наименование
 100.0  Описание инцидента
 100.0  Легитимность действия
 100.0  Инцидент был изменен
 100.0  SLA по устранению Атаки
 100.0  SLA по взятию в работу Атаки
 100.0  verdict
  99.9  Критичность инцидента в источнике
  99.9  Статус инцидента из SIEM
  99.9  Инцидент в Источнике
  99.9  Дата создания в источнике
  99.9  Дата изменения в источнике
  99.8  Период взятия в работу
  99.6  Ответственный
  99.4  Ответсвенный по РП
  99.3  Время решения
  96.6  Значение Атаки (группировка)
  96.2  Значение Инцидента (группировка)
  95.0  Принятые меры
  94.5  Правило корреляции SIEM
  88.3  Значение Инцидента (группировка по типу)
  85.9  Решение
  83.1  задержка перед автоназначением
  82.9  Классификация выполнена
  82.6

In [17]:
%run etl.py --full

Читаю iavd_incidents_raw_target.parquet: 240,295 строк, row groups: 3
  row group 1/3: 100,000 строк
  row group 2/3: 100,000 строк
  row group 3/3: 40,295 строк

=== Готово ===
Период: 2024-04-12 20:06:04.854999 — 2025-09-03 15:21:17.348999
Fact_label (-1 другое, 0 FP, 1 реальный):
y
-1     61014
 0    150675
 1     28606
Доля положительного класса среди размеченных: 0.160

Заполненность числовых признаков, %:
lag_min           94.7
attack_dur_min    82.6
cmd_len           29.3
n_incidents       16.9
react_min         88.6
solve_min         91.0

react_min / solve_min (минуты), квантили:
      react_min  solve_min
0.50        2.2        1.5
0.90       86.8       41.9
0.99     2239.5     1703.9


In [18]:
%run train.py

train: 134,460 (доля 1 = 0.185) | test: 44,821 (доля 1 = 0.083) | граница по времени: 2025-03-05 15:56:25.675000

ТЕСТ: ROC AUC = 0.870 | PR AUC = 0.433 (случайная модель: PR AUC = 0.083)

Топ-10 признаков по вкладу (gain, %):
rule_c        18.5
cat_main_c    14.8
cat_low_c     11.9
lag_min       10.8
desc_len       4.9
obj_c          4.7
dow            4.6
hour           4.4
source_c       4.4
inc_type_c     3.8

Bias-variance (log-loss):
 max_depth  train_loss  val_loss
         1       0.434     0.341
         2       0.415     0.350
         3       0.396     0.363
         4       0.376     0.360
         6       0.331     0.345
         8       0.282     0.338
        10       0.238     0.336
        12       0.218     0.320

Готово: файлы в /Users/lidakarpovich/intelligent-data-analysis-and-visualization/data/agg


In [2]:
from taipy.gui import Gui
import plotly.express as px
import pandas as pd

# --- заглушка: потом заменим на данные из data/agg ---
demo = pd.DataFrame({"bin": range(10), "count": [5, 9, 14, 20, 17, 12, 8, 5, 3, 1]})
fig_demo = px.bar(demo, x="bin", y="count", title="Заглушка")

root_page = """
<|navbar|>
"""

overview_page = """
# Обзор
KPI и фильтры появятся здесь.
"""

eda_page = """
# EDA и распределения
<|chart|figure={fig_demo}|>
"""

model_page = """
# Качество модели
ROC, PR, порог, матрица ошибок.
"""

xai_page = """
# Объяснимый ИИ
SHAP, PDP, bias-variance.
"""

advanced_page = """
# Альтернативные визуализации
Hexbin, Sankey, Dot Plot, Bubble.
"""

pages = {
    "/": root_page,
    "overview": overview_page,
    "eda": eda_page,
    "model": model_page,
    "xai": xai_page,
    "advanced": advanced_page,
}

if __name__ == "__main__":
    Gui(pages=pages).run(title="SOC dashboard", dark_mode=False, port=5001)

[2026-10-23 18:21:56.851][Taipy][INFO] Running in 'single_client' mode in notebook environment
[2026-10-23 18:21:57.015][Taipy][INFO]  * Server starting on http://127.0.0.1:5001
